In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [5]:
abc = pd.read_csv("../data/cleaned/final_feature_engineered.csv")

In [7]:
abc.head()

,Store,Dept,Weekly_Sales,IsHoliday_x,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,year,month,Quarter,week,DayOfWeek
0,1,1,24924.50,0,0,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,2010,2,1,5,4
1,1,1,46039.49,1,0,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,2010,2,1,6,4
2,1,1,41595.55,0,0,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,2010,2,1,7,4
3,1,1,19403.54,0,0,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,2010,2,1,8,4
4,1,1,21827.90,0,0,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,2010,3,1,9,4


In [9]:
X=abc.drop("Weekly_Sales",axis=1)
y=abc["Weekly_Sales"]

In [11]:
X_train,X_text,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [16]:
print(X_train.shape)
print(X_text.shape)

print(y_train.shape)
print(y_test.shape)

(337256, 19)
(84314, 19)
(337256,)
(84314,)


In [19]:
model=LinearRegression()

In [21]:
model.fit(X_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [23]:
y_predict=model.predict(X_text)

In [26]:
print(y_predict[:5])

[19534.67998434 20814.40427756  2183.89926794 10756.16284237
 25524.68456837]


In [28]:
mae=mean_absolute_error(y_test,y_predict)
mse=mean_squared_error(y_test,y_predict)
rmse=np.sqrt(mse)
r2=r2_score(y_test,y_predict)


In [29]:
print("MAE :", mae)

print("MSE :", mse)

print("RMSE :", rmse)

print("R2 Score :", r2)

MAE : 14551.750602278937
MSE : 474815473.85285527
RMSE : 21790.260986341014
R2 Score : 0.0894690589929722


linear regreesion cant be deployed as r-square vlaue is very poor, aslo mae error is too high, model is making huge errores 

In [32]:
tree_model=DecisionTreeRegressor(random_state=42)

In [34]:
tree_model.fit(X_train,y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [37]:
tree_pred= tree_model.predict(X_text)

In [39]:
tree_mae = mean_absolute_error(y_test, tree_pred)

tree_mse = mean_squared_error(y_test, tree_pred)

tree_rmse = np.sqrt(tree_mse)

tree_r2 = r2_score(y_test, tree_pred)

print("MAE :", round(tree_mae,2))
print("RMSE :", round(tree_rmse,2))
print("R2 :", round(tree_r2,4))

MAE : 1874.59
RMSE : 4895.44
R2 : 0.954


In [42]:
print(abc["Weekly_Sales"].mean())

15981.258123467042


In [45]:
print(tree_model.score(X_train,y_train))
print(tree_model.score(X_text,y_test))

1.0
0.9540428557082706


now the final random forest regressor

In [53]:
rf_model=RandomForestRegressor(n_estimators=50,random_state=42)

In [60]:
rf_model.fit(X_train,y_train)

,n_estimators,50
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [62]:
rf_pred=rf_model.predict(X_text)


In [64]:
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("MAE :", round(rf_mae, 2))
print("RMSE :", round(rf_rmse, 2))
print("R² :", round(rf_r2, 4))

MAE : 1453.13
RMSE : 3706.14
R² : 0.9737


In [66]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)
importance

,Feature,Importance
1,Dept,0.625632
4,Size,0.191335
0,Store,0.056921
17,week,0.041874
12,CPI,0.026455
3,Type,0.014554
13,Unemployment,0.010888
5,Temperature,0.010284
15,month,0.005784
6,Fuel_Price,0.005100


In [68]:
import joblib

In [70]:
joblib.dump(
    rf_model,
    "../models/walmart_sales_model.pkl"
)

['../models/walmart_sales_model.pkl']

#BUSINESS INSIGHTS.


1. Department is the most influential feature in predicting weekly sales, contributing approximately 62.6% of the model's predictive importance. This indicates that demand varies significantly across different product departments.

2. Store Size is the second most important feature (19.1%), indicating that larger stores generally generate higher weekly sales due to greater customer traffic and product variety.

3. The Week feature contributes approximately 4.2% to the prediction, suggesting that customer demand changes throughout the year because of seasonal purchasing patterns.

4. The Store feature contributes approximately 5.7% to sales prediction, indicating that sales vary 
across different Walmart locations due to regional customer demand and local buying behavior.

5. Store Type (A, B, and C) contributes to prediction accuracy, indicating that different store formats exhibit distinct customer demand patterns.

6. The MarkDown (Promotional) variables contributed less to prediction than expected. 
This suggests that promotions alone were not the primary drivers of weekly sales within this dataset.

7. The IsHoliday feature showed relatively low importance. 
This indicates that holidays alone were not sufficient to explain sales variation.

8. Random Forest achieved the highest prediction accuracy (R² = 0.974), explaining approximately 97.4% of the variation in weekly sales. This outperformed both Linear Regression and Decision Tree, making it the final selected model.

9. The top four features (Department, Store Size, Store, and Week) together account for over 90% of the model's predictive importance, indicating that product category, store characteristics, and seasonal demand are the primary drivers of Walmart's weekly sales.


